# UC-11 · Mechanical running test acceptance

**Use case:** [UC-11](../docs/product/use-cases.md) - *a 4 h mechanical run is logged
every 15 min; accept or reject it.*

`pump.mechanical_run` models a time-stamped log of channels, each with a **role**
(`vibration_filtered`, `bearing_temperature`, ...), a unit chosen by the engineer, and
a checker that compares each channel with a criterion of its role:

- an upper limit on the channel maximum;
- an optional *stabilisation* rule: over the last window, max - min must stay below a
  change (temperature differences use `delta_degF`);
- the minimum run duration.

A check without a configured criterion reports `info`, never pass or fail, and the
verdict then reads *PASSED (not fully assessed)*.

> The shipped limits (4 h, 2.0 / 3.0 mm/s RMS) are **unverified** seeds.  The second
> criteria set in this notebook is an *illustrative demo* set, not a standard.

**Acceptance:** the demo run passes against the shipped criteria (not fully assessed)
and against the demo criteria; a vibration spike, a short run and an unstable oil
temperature each fail.

## 1 · Setup

In [ ]:
import math

import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import (
    AcceptanceCriteria,
    Channel,
    MechanicalRunChecker,
    MechanicalRunCriteria,
    MechanicalRunLog,
    Q_,
    RunReading,
    plot_mechanical_run,
)

## 2 · A 4 h log in the engineer's units

Vibration in in/s, temperatures in degF, one reading every 15 min from 10:00 to 14:00
(17 readings).  Options inject a vibration spike, shorten the run or keep the oil
temperature rising.

In [ ]:
CHANNELS = (
    Channel("Filtered vibration", "vibration_filtered"),
    Channel("Unfiltered vibration", "vibration_unfiltered"),
    Channel("Bearing temperature", "bearing_temperature"),
    Channel("Oil temperature", "oil_temperature"),
)


def make_log(n=17, spike=None, oil_ramp=False):
    readings = []
    for i in range(n):
        minutes = 15 * i
        oil = 100.0 + 1.5 * i if oil_ramp else 118.0 + 10.0 * (1.0 - math.exp(-i / 2.5))
        unfiltered = 0.16 if spike == i else 0.09
        readings.append(
            RunReading(
                f"{10 + minutes // 60:02d}:{minutes % 60:02d}",
                {
                    "Filtered vibration": Q_(0.05, "in/s"),
                    "Unfiltered vibration": Q_(unfiltered, "in/s"),
                    "Bearing temperature": Q_(150.0 + 0.5 * i, "degF"),
                    "Oil temperature": Q_(oil, "degF"),
                },
            )
        )
    return MechanicalRunLog(CHANNELS, readings)


log = make_log()
print("readings:", len(log.readings), "| duration:", log.duration.to("h"))

## 3 · Against the shipped criteria (API 610 12th seeds)

Vibration limits and the 4 h duration are configured; temperatures are not, so those
channels are `info` and the run is *not fully assessed*.

In [ ]:
shipped = AcceptanceCriteria.load("api610:12").mechanical_run
res_shipped = MechanicalRunChecker(log, shipped).check()
print("verdict:", res_shipped.verdict, "| verified criteria:", res_shipped.verified)
for c in res_shipped.channels:
    print(f"  {c.name:<22} max {c.max.magnitude:7.2f} {c.unit:<6} limit {c.limit} -> {c.status}")

## 4 · Against an illustrative full criteria set

Adds a bearing-temperature maximum of 95 degC and an oil-temperature stabilisation rule
(change <= 2 delta_degF over the last 60 min).

In [ ]:
demo = MechanicalRunCriteria(
    min_duration=Q_(4, "h"),
    role_max={
        "vibration_filtered": Q_(2.0, "mm/s"),
        "vibration_unfiltered": Q_(3.0, "mm/s"),
        "bearing_temperature": Q_(95, "degC"),
    },
    stabilisation={"oil_temperature": (Q_(2, "delta_degF"), Q_(60, "min"))},
    source="illustrative demo limits",
    verified=False,
)
res_ok = MechanicalRunChecker(log, demo).check()
print("verdict:", res_ok.verdict)
for c in res_ok.channels:
    print(f"  {c.name:<22} {c.status:<5} stabilisation: {c.stabilisation and c.stabilisation['ok']}")

png = plot_mechanical_run(log, demo)
display(Image(data=png.getvalue()))

## 5 · Three ways to fail

A single unfiltered-vibration spike of 0.16 in/s (4.06 mm/s), a run that stops after 2 h,
and an oil temperature that is still rising at the end of the run.

In [ ]:
res_spike = MechanicalRunChecker(make_log(spike=8), demo).check()
res_short = MechanicalRunChecker(make_log(n=9), demo).check()
res_ramp = MechanicalRunChecker(make_log(oil_ramp=True), demo).check()

for name, res in (("spike", res_spike), ("short run", res_short), ("oil still rising", res_ramp)):
    failing = [c.name for c in res.channels if c.status == "fail"]
    print(f"{name:<18} {res.verdict:<8} duration {res.duration_status:<5} failing channels: {failing}")

print("4.06 mm/s check:", Q_(0.16, "in/s").to("mm/s"))

## 6 · Acceptance

In [ ]:
# the demo log is 4 h long
assert abs(log.duration.to("h").magnitude - 4.0) < 1e-9
assert len(log.readings) == 17

# shipped criteria: nothing fails, but the temperatures are not assessed
assert res_shipped.passed and res_shipped.verdict == "PASSED (not fully assessed)"
assert res_shipped.fully_assessed is False
assert res_shipped.verified is False

# demo criteria: fully assessed and passed (still unverified criteria)
assert res_ok.passed and res_ok.fully_assessed and res_ok.verdict == "PASSED"
by_name = {c.name: c for c in res_ok.channels}
assert by_name["Oil temperature"].stabilisation["ok"] is True
assert by_name["Oil temperature"].status == "pass"

# failures
assert res_spike.verdict == "FAILED"
assert {c.name: c.status for c in res_spike.channels}["Unfiltered vibration"] == "fail"
assert res_short.verdict == "FAILED" and res_short.duration_status == "fail"
assert res_ramp.verdict == "FAILED"
assert {c.name: c.status for c in res_ramp.channels}["Oil temperature"] == "fail"
assert abs(Q_(0.16, "in/s").to("mm/s").magnitude - 4.064) < 1e-3
print("UC-11 acceptance OK")